# Benchmark STT tunisien — Whisper Large-v3 code-switch + LoRA

**Modèle évalué :** `oddadmix/Whisperv3-tunisian-codeswitch` + l'adapter `best_adapter` entraîné pendant le stage.

Ce notebook reprend exactement le protocole du notebook baseline : même ZIP WhatsApp figé, même association audio/référence, normalisation limitée aux espaces, WER, CER, latence, RTF, VRAM et exports CSV/JSON.

Le ZIP de test externe ne doit jamais avoir été utilisé pendant l'entraînement, la validation ou le choix des hyperparamètres. Le notebook charge les poids baseline gelés puis applique l'adapter LoRA final ; il ne lance aucun entraînement.

Le décodage reste reproductible : langue arabe (`ar`), transcription, décodage glouton (`num_beams=1`, sans échantillonnage) et sans timestamps.

In [ ]:
# Exécuter cette cellule avant les autres. Compatible Google Colab.
%pip install -q --no-cache-dir --upgrade 'transformers==4.48.3' 'tokenizers>=0.20' 'huggingface-hub>=0.23.2'
%pip install -q --no-cache-dir 'peft==0.14.0' 'accelerate>=1.0' 'jiwer==3.1.0' 'librosa==0.10.2.post1' 'soundfile==0.13.1' 'psutil>=5.9'

import csv
import json
import platform
import re
import shutil
import time
import zipfile
from pathlib import Path
from statistics import median

import librosa
import psutil
import torch
from jiwer import cer, process_characters, process_words, wer
from peft import PeftModel
from transformers import AutoModelForSpeechSeq2Seq, WhisperProcessor

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MODEL_ID = 'oddadmix/Whisperv3-tunisian-codeswitch'
PROCESSOR_ID = 'openai/whisper-large-v3'
TARGET_SR = 16000
WORKDIR = Path('/content/whisperv3_tunisian_codeswitch_lora_benchmark') if IN_COLAB else Path.cwd() / 'whisperv3_tunisian_codeswitch_lora_benchmark'
DATA_DIR = WORKDIR / 'data'
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
DATA_DIR.mkdir(parents=True)
PROCESS = psutil.Process()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
print('ENVIRONNEMENT')
print('Device :', DEVICE)
print('RAM disponible :', round(psutil.virtual_memory().available / 2**30, 2), 'GiB')

In [ ]:
# Téléverser le MÊME ZIP WhatsApp externe que pour le benchmark baseline.
if IN_COLAB:
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    assert zip_name is not None, 'Aucun fichier ZIP de test sélectionné.'
else:
    DATASET_ZIP = Path('whatsapp_test_set.zip')
    assert DATASET_ZIP.exists(), 'Dépose le ZIP puis adapte DATASET_ZIP dans cette cellule.'
    zip_name = str(DATASET_ZIP)

def extract_safely(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f'Archive non sûre : {member.filename}')
        archive.extractall(destination)

extract_safely(zip_name, DATA_DIR)
reference_files = list(DATA_DIR.rglob('references_transcriptions.txt'))
AUDIO_EXTENSIONS = {'.wav', '.ogg', '.mp3', '.flac', '.m4a'}
audio_files = sorted(
    [path for path in DATA_DIR.rglob('*') if path.suffix.lower() in AUDIO_EXTENSIONS],
    key=lambda path: path.name,
)
assert len(reference_files) == 1, 'references_transcriptions.txt est absent ou dupliqué.'
references = [line.strip() for line in reference_files[0].read_text(encoding='utf-8').splitlines()]
assert len(audio_files) == len(references), f'{len(audio_files)} audios trouvés, mais {len(references)} transcriptions.'
assert all(references), 'Une transcription de référence est vide.'
samples = [{'id': audio.stem, 'audio_file': audio, 'reference': reference} for audio, reference in zip(audio_files, references)]
assert samples, 'Le ZIP ne contient aucun audio compatible.'
print(f'TEST SET EXTERNE VALIDÉ : {len(samples)} audios')
print('Premier audio :', samples[0]['audio_file'].name)

In [ ]:
# Charger la baseline puis l'adapter LoRA FINAL (best_adapter).
# Colab : téléverser lora_full_outputs.zip dans cette cellule.
if IN_COLAB:
    uploaded_adapter = files.upload()
    adapter_archive_name = next((name for name in uploaded_adapter if name.lower().endswith('.zip')), None)
    assert adapter_archive_name is not None, 'Téléverse lora_full_outputs.zip.'
    adapter_archive = Path(adapter_archive_name)
else:
    adapter_archive = Path('lora_full_outputs.zip')
    assert adapter_archive.exists(), 'Place lora_full_outputs.zip à côté du notebook ou adapte ce chemin.'

def extract_best_adapter(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        members = [member for member in archive.infolist() if member.filename.startswith('best_adapter/')]
        assert members, 'best_adapter/ est absent de l’archive LoRA.'
        for member in members:
            target = (destination / member.filename).resolve()
            if destination not in target.parents:
                raise ValueError(f'Archive non sûre : {member.filename}')
            archive.extract(member, destination)

extract_best_adapter(adapter_archive, WORKDIR)
ADAPTER_DIR = WORKDIR / 'best_adapter'
assert (ADAPTER_DIR / 'adapter_model.safetensors').exists(), 'Poids LoRA introuvables.'
assert (ADAPTER_DIR / 'adapter_config.json').exists(), 'Configuration LoRA introuvable.'

model_load_start = time.perf_counter()
# Même processor Whisper Large-v3 que le benchmark baseline : protocole comparable.
processor = WhisperProcessor.from_pretrained(PROCESSOR_ID)
base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False).to(DEVICE)
model.eval()
model_load_s = time.perf_counter() - model_load_start

FORCED_DECODER_IDS = processor.get_decoder_prompt_ids(
    language='ar',
    task='transcribe',
    no_timestamps=True,
)

if DEVICE == 'cuda':
    torch.cuda.synchronize()
    model_vram_mib = torch.cuda.memory_allocated() / 2**20
else:
    model_vram_mib = None

print('BASELINE :', MODEL_ID)
print('ADAPTER LoRA :', ADAPTER_DIR)
print('Temps de chargement :', round(model_load_s, 2), 's')

In [ ]:
def normalize_for_scoring(text):
    # Même normalisation minimale que le benchmark baseline : espaces uniquement.
    return re.sub(r'\s+', ' ', str(text)).strip()

def load_audio(path):
    if not Path(path).exists():
        raise FileNotFoundError(f'Fichier introuvable : {path}')
    audio, _ = librosa.load(str(path), sr=TARGET_SR, mono=True)
    return audio, len(audio) / TARGET_SR

def transcribe(path):
    audio, duration = load_audio(path)
    inputs = processor(audio, sampling_rate=TARGET_SR, return_tensors='pt')
    input_features = inputs.input_features.to(device=DEVICE, dtype=DTYPE)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        generated_ids = model.generate(
            input_features=input_features,
            forced_decoder_ids=FORCED_DECODER_IDS,
            num_beams=1,
            do_sample=False,
            max_new_tokens=444,
        )
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    latency = time.perf_counter() - start
    prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return normalize_for_scoring(prediction), duration, latency

results = []
peak_rss_bytes = PROCESS.memory_info().rss
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()

for position, sample in enumerate(samples, start=1):
    audio_path = sample['audio_file']
    prediction, duration, latency = transcribe(audio_path)
    peak_rss_bytes = max(peak_rss_bytes, PROCESS.memory_info().rss)
    row = {
        'id': sample['id'],
        'audio_path': audio_path.relative_to(DATA_DIR).as_posix(),
        'reference': normalize_for_scoring(sample['reference']),
        'prediction': prediction,
        'duration_s': duration,
        'latency_s': latency,
        'rtf': latency / duration if duration else None,
    }
    results.append(row)
    print(f'[{position:>3}/{len(samples)}] {row["id"]}: {duration:.2f}s, {latency:.3f}s, RTF={row["rtf"]:.3f}')

assert results, 'Aucun résultat produit. Vérifiez la structure du fichier ZIP.'

In [ ]:
# Métriques et exports : même protocole que la baseline, avec identification LoRA.
reference_texts = [row['reference'] for row in results]
prediction_texts = [row['prediction'] for row in results]
word_details = process_words(reference_texts, prediction_texts)
character_details = process_characters(reference_texts, prediction_texts)
total_audio_s = sum(row['duration_s'] for row in results)
total_inference_s = sum(row['latency_s'] for row in results)
peak_vram_mib = torch.cuda.max_memory_allocated() / 2**20 if DEVICE == 'cuda' else None

summary = {
    'model_id': MODEL_ID,
    'model_status': 'community baseline + LoRA adapter trained during the internship',
    'adapter': 'best_adapter from lora_full_outputs.zip',
    'dataset': 'user-selected external WhatsApp ZIP test set',
    'input_preprocessing': 'audio decoded to mono 16 kHz',
    'decoding': 'language=ar; task=transcribe; num_beams=1; do_sample=false; no_timestamps=true',
    'normalization': 'whitespace only',
    'n_utterances': len(results),
    'total_audio_s': total_audio_s,
    'wer_percent': 100 * wer(reference_texts, prediction_texts),
    'cer_percent': 100 * cer(reference_texts, prediction_texts),
    'word_substitutions': word_details.substitutions,
    'word_deletions': word_details.deletions,
    'word_insertions': word_details.insertions,
    'character_substitutions': character_details.substitutions,
    'character_deletions': character_details.deletions,
    'character_insertions': character_details.insertions,
    'mean_latency_s': total_inference_s / len(results),
    'median_latency_s': median(row['latency_s'] for row in results),
    'total_inference_s': total_inference_s,
    'mean_rtf': sum(row['rtf'] for row in results) / len(results),
    'corpus_rtf': total_inference_s / total_audio_s,
    'peak_process_rss_mib': peak_rss_bytes / 2**20,
    'model_resident_vram_mib': model_vram_mib,
    'peak_vram_mib': peak_vram_mib,
    'model_load_s': model_load_s,
    'device': DEVICE,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
    'torch_version': torch.__version__,
    'transformers_version': __import__('transformers').__version__,
    'peft_version': __import__('peft').__version__,
    'python_version': platform.python_version(),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

output_csv = WORKDIR / 'whisperv3_tunisian_codeswitch_lora_whatsapp_predictions.csv'
output_json = WORKDIR / 'whisperv3_tunisian_codeswitch_lora_whatsapp_summary.json'
with output_csv.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)
with output_json.open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)
print('EXPORTS CRÉÉS :', output_csv.name, 'et', output_json.name)
if IN_COLAB:
    files.download(str(output_csv))
    files.download(str(output_json))

## Interprétation

Compare ce modèle uniquement à la baseline exécutée sur le même ZIP externe, avec la même normalisation et les mêmes paramètres de décodage. Toute différence de WER/CER peut alors être attribuée à l'adapter LoRA, dans les limites de la variabilité de l'environnement matériel.

Le CSV permet une analyse qualitative des prédictions ; le JSON contient les métriques globales et la configuration nécessaire au rapport de stage. Le test final reste exclusivement une évaluation finale : ne l'utilise pas pour modifier l'adapter ou relancer l'entraînement.